# Argument realization in an Ojibwe corpus: parsing the treebank

In this notebook, we will take an `.xml` formatted corpus file and turn it into a `.conllu` formatted treebank.

### Part 1: Parsing the xml file
We will begin by taking a parsed Ojibwe text in `.xml` format (currently using section 4 of Living our Language), and storing Ojibwe and English sentence pairs in a list variable `sentences`. 

**N.B.** The Ojibwe and English sentences in each paragraph are assumed to stand in a one-to-one relationship, which is admittedly a coarse heuristic, meaning there may be mismatches. The English sentences are primarily for reference when working with the treebank data.

In [ ]:
from grammar_modules.disambiguation import REPO_ROOT
import xml.etree.ElementTree as ET
import re

XML_PATH = REPO_ROOT / "data" / "corpus" / "Living_our_Language" / "LoL_section4.xml"

tree = ET.parse(XML_PATH) 
root = tree.getroot()

sentence_count = 0
sentences = [] # tuple of matching (oj, en) sentence pairs
en_sents = 0
for para in root.findall(".//paragraph"):
    eng_para = para.find("text_eng").text
    eng_sentences = re.split(r'(?<=[.!?])\s*', eng_para)
    eng_sentences = [s for s in eng_sentences if s]

    for i, sent in enumerate(para.findall("text_ojb/sentence")):
        sentence_count+=1
        # find sent_text node
        sent_text_node = sent.find("sent_text")
        # get actual sentence text
        oj_sent_text = sent_text_node.text.strip() if sent_text_node.text is not None else None
        if oj_sent_text is None: continue
        if len(eng_sentences) > i: 
            en_sent_text = eng_sentences[i]
        else: 
            en_sent_text = "No English translation."

        sentences.append((oj_sent_text, en_sent_text))
        print(f"#", sentence_count, "Oj text:", oj_sent_text, "En text:", en_sent_text)
        print("-" * 40)




# 1 Oj text: Zhaawanoowinini indizhinikaaz, miinawaa dash a'aw ogiishkimanisii indoodem. En text:  My name is Zhaawanoowinini, and my clan is the Kingfisher.
----------------------------------------
# 2 Oj text: Imaa wenjibaayaan, imaa Miskwaagamiwi-zaaga'iganiing, mii wenjiiwaad ingitiziimag apane. En text: Where I am from, there at Red Lake, that’s where my parents were from.
----------------------------------------
# 3 Oj text: Miinawaa dash a'aw nimaamaayiban, onow odoodeman migiziwan. En text: And my late mother, she was of the Bald Eagle Clan.
----------------------------------------
# 4 Oj text: Ganabaj a'aw nimishoomisiban Zhaaganaashiiwakiing gii-onjibaa. En text: My grandfather may have been from Canada.
----------------------------------------
# 5 Oj text: Gii-pi-izhaa omaa. En text: He came here.
----------------------------------------
# 6 Oj text: Aabiding igo ogii-mawidisaan onow ikwewan imaa Obaashiing. En text: One time he visited this woman there at Ponema.
----------

### Part 2: Running the treebank pipeline

We now run each sentence through the dependency parsing pipeline to create a `.conllu` formatted treebank. The xml contains disambiguated FST readings, but we will re-run sentences through the pipeline, so that we keep the parsing local to this repository.

In [4]:
from grammar_modules.dependency import DEPENDENCY_PATH, parse_dependencies
from treebank_modules.corpus import cg3_to_conllu_batch
from grammar_modules.disambiguation import DISAMBIGUATION_PATH
from grammar_modules.fst import load_fst_parser

# now we actually build up the .conllu file by first parsing dependencies on each sentence,
# and then appending it to the corpus file
CORPUS_PATH = REPO_ROOT / "data" / "treebanks" / "LoL_section4_with_eng.conllu"

# fst + grammar paths
FST = load_fst_parser()
DISAMBIG_CG_PATH = REPO_ROOT / "data" / "rules" / "disambiguation.cg3"
DEPENDENCY_CG_PATH = REPO_ROOT / "data" / "rules" / "dependency.cg3"


num_sents = 0

# TODO: certain sentences repeat in the text, but they are only stored once in the corpus. 
#       These are definitely relevant to accurate statistics, should I just remove duplicate
#       sentence logic in cg3_to_conllu_batch() ?  
# for each sentence, parse dependencies and append to conllu corpus
for sent in sentences:
    oj_line = sent[0]
    en_line = sent[1]

    # parse the deps
    dep_cg3 = parse_dependencies(
       sentence=oj_line,
        dependency_grammar=str(DEPENDENCY_PATH),
        disambiguation_grammar=str(DISAMBIGUATION_PATH),
        fst=FST,
        verbose=False
    )
    # convert and append to corpus (auto sent_id)
    cg3_to_conllu_batch(dep_cg3, corpus_path=str(CORPUS_PATH), en_line=en_line)
    num_sents += 1

print(f"Added {num_sents} sentences to treebank.")

FST file is /Users/matthias/labs/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin
✓ appended sentence #1 to LoL_section4_with_eng.conllu
✓ appended sentence #2 to LoL_section4_with_eng.conllu
✓ appended sentence #3 to LoL_section4_with_eng.conllu
✓ appended sentence #4 to LoL_section4_with_eng.conllu
✓ appended sentence #5 to LoL_section4_with_eng.conllu
✓ appended sentence #6 to LoL_section4_with_eng.conllu
✓ appended sentence #7 to LoL_section4_with_eng.conllu
✓ appended sentence #8 to LoL_section4_with_eng.conllu
✓ appended sentence #9 to LoL_section4_with_eng.conllu
✓ appended sentence #10 to LoL_section4_with_eng.conllu
✓ appended sentence #11 to LoL_section4_with_eng.conllu
✓ appended sentence #12 to LoL_section4_with_eng.conllu
✓ appended sentence #13 to LoL_section4_with_eng.conllu
✓ appended sentence #14 to LoL_section4_with_eng.conllu
✓ appended sentence #15 to LoL_section4_with_eng.conllu
✓ appended sentence #16 to LoL_section4_with_eng.conllu
✓ appended sent